# 01 — Dataset, embeddings and similarity retrieval (T-011/S3)

Components verified: catalog dataset, embedding model, exact cosine top-N retrieval and the ONNX inference path the Java backend will mirror (`docs/AI_MVP_DESIGN.md` §3–5, `docs/NOTEBOOK_VALIDATION.md`).

**Relevance proxy.** A catalog product is relevant to a query product when both share the same Open Food Facts leaf category (`main_category_en`). Because the production text contains category words, the *leak-free benchmark* uses a `no_category` text (name, brand, description only); `full`-text scores are reported but inflated by construction.

In [1]:
import json, hashlib, platform, random, time
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch, sklearn, sentence_transformers, onnxruntime
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
import os, sys; os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from ml.asap_ml import retrieval as R
OUT = ROOT / 'notebooks/results/01_data_embeddings_retrieval'; OUT.mkdir(parents=True, exist_ok=True)
SEED = 20261006; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.set_num_threads(8)
manifest = json.loads(Path('data/catalog_manifest.json').read_text())
assert hashlib.sha256(Path('data/processed/catalog.jsonl').read_bytes()).hexdigest() == manifest['catalog_sha256']
env = {'python': platform.python_version(), 'torch': torch.__version__, 'sentence_transformers': sentence_transformers.__version__,
       'sklearn': sklearn.__version__, 'onnxruntime': onnxruntime.__version__, 'cpu': platform.processor() or platform.machine(),
       'threads': torch.get_num_threads(), 'run_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
env

{'python': '3.13.0',
 'torch': '2.14.1+cpu',
 'sentence_transformers': '6.1.0',
 'sklearn': '1.9.1',
 'onnxruntime': '1.30.0',
 'cpu': 'x86_64',
 'threads': 8,
 'run_utc': '2026-10-06T12:12:19Z'}

## 1. Dataset

In [2]:
cat = R.load_catalog(); N = len(cat)
df = pd.DataFrame({'source': [p['source'] for p in cat], 'category': [p['category'] for p in cat],
                   'top': [p['top_category'] for p in cat], 'has_brand': [bool(p['brand']) for p in cat],
                   'has_desc': [bool(p['description']) for p in cat], 'region': [p['region'] for p in cat],
                   'len_full': [len(R.product_text(p)) for p in cat], 'len_nocat': [len(R.product_text(p, 'no_category')) for p in cat]})
cat_sizes = df.category.value_counts()
stats = {'products': N, 'by_source': df.source.value_counts().to_dict(), 'top_categories': df.top.nunique(),
         'leaf_categories': int(cat_sizes.size), 'leaf_categories_ge5': int((cat_sizes >= 5).sum()),
         'share_brand': round(df.has_brand.mean(), 3), 'share_description': round(df.has_desc.mean(), 3),
         'share_region': round(df.region.mean(), 3), 'median_text_len_full': int(df.len_full.median()),
         'median_text_len_no_category': int(df.len_nocat.median())}
stats

{'products': 10000,
 'by_source': {'open_food_facts': 8000,
  'open_beauty_facts': 1200,
  'open_pet_food_facts': 800},
 'top_categories': 46,
 'leaf_categories': 2143,
 'leaf_categories_ge5': 431,
 'share_brand': np.float64(0.979),
 'share_description': np.float64(0.321),
 'share_region': np.float64(0.648),
 'median_text_len_full': 116,
 'median_text_len_no_category': 38}

In [3]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
df.top.value_counts().head(20).iloc[::-1].plot.barh(ax=ax[0], title='Top-level strata (20 largest)')
ax[1].hist(cat_sizes.values, bins=np.logspace(0, np.log10(cat_sizes.max()), 25)); ax[1].set_xscale('log')
ax[1].set_title('Leaf-category size distribution'); ax[1].set_xlabel('products per leaf category')
plt.tight_layout(); plt.savefig(OUT / 'dataset.png', dpi=150); plt.show()

/tmp/ipykernel_184140/526734504.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUT / 'dataset.png', dpi=150); plt.show()


## 2. Query set
500 seeded queries drawn from leaf categories with ≥ 5 products, so every query has ≥ 4 relevant items.

In [4]:
groups_by_cat = defaultdict(set)
for i, p in enumerate(cat): groups_by_cat[p['category']].add(i)
groups = {i: groups_by_cat[p['category']] for i, p in enumerate(cat)}
eligible = [i for i in range(N) if len(groups[i]) >= 5]
rng = random.Random(SEED)
queries = sorted(rng.sample(eligible, 500))
top_groups = defaultdict(set)
for i, p in enumerate(cat): top_groups[p['top_category']].add(i)
coarse = {i: top_groups[p['top_category']] for i, p in enumerate(cat)}
random_p10 = float(np.mean([(len(groups[q]) - 1) / (N - 1) for q in queries]))
print(len(eligible), 'eligible;', 'random-ranking expected P@10 =', round(random_p10, 4))

7260 eligible; random-ranking expected P@10 = 0.0046


## 3. Methods
Lexical baselines (TF-IDF word 1–2-grams and character 3–5-grams, cosine) versus the two candidate sentence-embedding models. Every method is scored on both text variants with identical queries and metrics.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
def eval_scores(score_fn, qs=queries, rel=groups, k=10, pool=50):
    rows = []
    for q in qs:
        s = score_fn(q); s[q] = -np.inf
        idx = np.argpartition(-s, pool)[:pool]; idx = idx[np.lexsort((idx, -s[idx]))]
        rows.append(R.metrics(idx, rel[q] - {q}, k, pool))
    return {m: float(np.mean([r[m] for r in rows])) for m in rows[0]}
texts = {v: [R.product_text(p, v) for p in cat] for v in ('no_category', 'full')}
results, vectors, encode_s = [], {}, {}
for v in texts:
    for name, kw in [('tfidf-word', dict(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
                     ('tfidf-char', dict(analyzer='char_wb', ngram_range=(3, 5), min_df=2, sublinear_tf=True))]:
        X = TfidfVectorizer(**kw).fit_transform(texts[v])
        r = eval_scores(lambda q: (X @ X[q].T).toarray().ravel()); results.append({'method': name, 'text': v, **r,
            'coarse_p@10': eval_scores(lambda q: (X @ X[q].T).toarray().ravel(), rel=coarse)['p@10']})
for m in R.MODELS:
    for v in texts:
        t = time.perf_counter(); vec, _ = R.encode(m, texts[v]); encode_s[(m, v)] = time.perf_counter() - t
        vectors[(m, v)] = vec; assert np.isfinite(vec).all() and np.allclose(np.linalg.norm(vec, axis=1), 1, atol=1e-4)
        results.append({'method': m, 'text': v, **R.evaluate(vec, queries, groups), 'coarse_p@10': R.evaluate(vec, queries, coarse)['p@10']})
res = pd.DataFrame(results).set_index(['text', 'method']).sort_index()
res.round(4)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

p@10  ndcg@10     mrr  recall@50  coarse_p@10
text        method                                                     
full        e5-small    0.4284   0.4796  0.6958     0.5031       0.7292
            minilm      0.5664   0.6304  0.8055     0.6566       0.8016
            tfidf-char  0.6950   0.7638  0.8877     0.8468       0.8616
            tfidf-word  0.7876   0.8602  0.9380     0.9091       0.8942
no_category e5-small    0.2458   0.2883  0.4987     0.2607       0.5172
            minilm      0.1732   0.2064  0.3971     0.1994       0.3922
            tfidf-char  0.2484   0.2906  0.5000     0.2784       0.5134
            tfidf-word  0.2284   0.2676  0.4743     0.2587       0.4842

In [6]:
ax = res.loc['no_category'][['p@10', 'ndcg@10', 'recall@50']].plot.bar(rot=0, figsize=(8, 3.5))
ax.axhline(random_p10, ls='--', c='grey', label='random P@10'); ax.legend(); ax.set_ylim(0, 1)
ax.set_title('Leak-free retrieval quality (no_category text, 500 queries)')
plt.tight_layout(); plt.savefig(OUT / 'retrieval_quality.png', dpi=150); plt.show()
pd.Series({f'{m}/{v}': round(s, 1) for (m, v), s in encode_s.items()}, name='catalog encode seconds (10k, CPU)')

/tmp/ipykernel_184140/2669772700.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUT / 'retrieval_quality.png', dpi=150); plt.show()


e5-small/no_category     48.6
e5-small/full           112.4
minilm/no_category       49.9
minilm/full             117.0
Name: catalog encode seconds (10k, CPU), dtype: float64

## 4. Uncertainty
Bootstrap 95 % confidence interval of nDCG@10 per method (no_category text, 1,000 resamples of queries).

In [7]:
def per_query_ndcg(vec): return np.array([R.metrics(R.top_k(vec, q, 50), groups[q] - {q})['ndcg@10'] for q in queries])
boot = {}
brng = np.random.default_rng(SEED)
for m in R.MODELS:
    pq = per_query_ndcg(vectors[(m, 'no_category')])
    bs = [pq[brng.integers(0, len(pq), len(pq))].mean() for _ in range(1000)]
    boot[m] = (pq.mean(), *np.percentile(bs, [2.5, 97.5]))
pd.DataFrame(boot, index=['ndcg@10', 'ci_low', 'ci_high']).T.round(4)

,ndcg@10,ci_low,ci_high
e5-small,0.2883,0.2628,0.3122
minilm,0.2064,0.1852,0.2285


## 5. Selection
Rule fixed before the run: highest leak-free nDCG@10 among the embedding models; it must also beat both TF-IDF baselines.

In [8]:
emb = res.loc['no_category'].loc[list(R.MODELS)]
chosen = emb['ndcg@10'].idxmax()
beats_tfidf = bool(emb.loc[chosen, 'ndcg@10'] > res.loc['no_category'].loc[['tfidf-word', 'tfidf-char'], 'ndcg@10'].max())
print('chosen:', chosen, '| beats TF-IDF:', beats_tfidf)

chosen: e5-small | beats TF-IDF: False


## 5b. Hybrid lexical + semantic fusion
Because the chosen embedding only ties character TF-IDF, test the standard remedy: `s = α·cos_e5 + (1−α)·cos_tfidf_char`. α is tuned on a **separate** 500-query tuning set (disjoint from the test queries) and then evaluated once on the test set with paired bootstrap CIs of the nDCG@10 difference.

In [9]:
tune_pool = sorted(set(eligible) - set(queries)); tune = sorted(random.Random(SEED + 1).sample(tune_pool, 500))
hyb = {}
for v in texts:
    Xv = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2, sublinear_tf=True).fit_transform(texts[v])
    Ev = vectors[(chosen, v)]
    hyb[v] = (Xv, Ev)
def hybrid_fn(v, a):
    Xv, Ev = hyb[v]
    return lambda q: a * (Ev @ Ev[q]) + (1 - a) * (Xv @ Xv[q].T).toarray().ravel()
alphas = np.round(np.arange(0, 1.01, 0.1), 2)
sweep = pd.DataFrame({v: [eval_scores(hybrid_fn(v, a), qs=tune)['ndcg@10'] for a in alphas] for v in texts}, index=alphas)
best_alpha = {v: float(sweep[v].idxmax()) for v in texts}
ax = sweep.plot(marker='o', figsize=(6, 3.5), title='Hybrid α sweep (tuning queries)'); ax.set_xlabel('α (weight of e5)'); ax.set_ylabel('nDCG@10')
plt.tight_layout(); plt.savefig(OUT / 'hybrid_alpha_sweep.png', dpi=150); plt.show(); best_alpha

/tmp/ipykernel_184140/2954568029.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUT / 'hybrid_alpha_sweep.png', dpi=150); plt.show(); best_alpha


{'no_category': 0.9, 'full': 0.0}

In [10]:
def per_query(score_fn, qs=queries):
    out = []
    for q in qs:
        sc = score_fn(q); sc[q] = -np.inf; idx = np.argpartition(-sc, 50)[:50]; idx = idx[np.lexsort((idx, -sc[idx]))]
        out.append(R.metrics(idx, groups[q] - {q})['ndcg@10'])
    return np.array(out)
v = 'no_category'; Xv, Ev = hyb[v]
pq = {'hybrid': per_query(hybrid_fn(v, best_alpha[v])), chosen: per_query(lambda q: Ev @ Ev[q]),
      'tfidf-char': per_query(lambda q: (Xv @ Xv[q].T).toarray().ravel())}
brng2 = np.random.default_rng(SEED + 2); idxs = [brng2.integers(0, len(queries), len(queries)) for _ in range(1000)]
def paired(a, b): d = pq[a] - pq[b]; bs = [d[i].mean() for i in idxs]; return [round(float(d.mean()), 4), *np.round(np.percentile(bs, [2.5, 97.5]), 4).tolist()]
hybrid_test = {'alpha': best_alpha, 'test_ndcg@10': {k: round(float(x.mean()), 4) for k, x in pq.items()},
               'diff_vs_' + chosen: paired('hybrid', chosen), 'diff_vs_tfidf-char': paired('hybrid', 'tfidf-char'),
               'full_text_test': eval_scores(hybrid_fn('full', best_alpha['full']))}
hybrid_wins = hybrid_test['diff_vs_' + chosen][1] > 0 and hybrid_test['diff_vs_tfidf-char'][1] > 0
print(json.dumps(hybrid_test, indent=1)); print('hybrid significantly beats both:', hybrid_wins)

{
 "alpha": {
  "no_category": 0.9,
  "full": 0.0
 },
 "test_ndcg@10": {
  "hybrid": 0.3066,
  "e5-small": 0.2883,
  "tfidf-char": 0.2906
 },
 "diff_vs_e5-small": [
  0.0183,
  0.0119,
  0.025
 ],
 "diff_vs_tfidf-char": [
  0.016,
  0.008,
  0.0245
 ],
 "full_text_test": {
  "p@10": 0.695,
  "ndcg@10": 0.7637658150177601,
  "mrr": 0.887716396469028,
  "recall@50": 0.8467798699144968
 }
}
hybrid significantly beats both: True


## 6. Qualitative check
Top-5 neighbours (production `full` text) for six seeded queries: chosen model vs. character TF-IDF.

In [11]:
Xc = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2, sublinear_tf=True).fit_transform(texts['full'])
vf = vectors[(chosen, 'full')]
rows = []
for q in rng.sample(queries, 6):
    s = (Xc @ Xc[q].T).toarray().ravel(); s[q] = -np.inf; t5 = np.argsort(-s)[:5]
    rows.append({'query': f"{cat[q]['name']} [{cat[q]['category']}]",
                 chosen: '; '.join(cat[i]['name'] for i in R.top_k(vf, q, 5)), 'tfidf-char': '; '.join(cat[i]['name'] for i in t5)})
pd.set_option('display.max_colwidth', 120); qual = pd.DataFrame(rows); qual

,query,e5-small,tfidf-char
0,Sertés májas [Pâté],"Nádudvari majoránnás májas; Sertésvirsli; Sertésmájas, finoman aprított; csemege debreceni kolbász; Prémium sertés c...","Nádudvari majoránnás májas; csemege debreceni kolbász; Sertésmájas, finoman aprított; Prémium sertés combsonka; Pate"
1,organic paté Lilys [Pastas],pate rotina; Sweat potato vermicelli; Patelina; Cetina; pate,"Spirali, PASTA ITALIANA; Fodros nagykocka; Terrine pour chien; pâté pour chat junior au saumon; Ditali grandi rigati"
2,merry Christmas [Dog biscuit],chew bites; Creamy Snack; Tuna Bites; Pedigree; Lean cuts,chew bites; Knotted chewing bones with chicken; sport snacks; chicko; pure duck
3,Spicy Corn Relish [Salted-snacks],New england cranberry relish; Farmhouse Red Relish; Tortilla Chips; Sweet pickle relish; No Frills Dills Pickles,Farmhouse Red Relish; New england cranberry relish; Apple Cranberry Chutney; Old Farmhouse Chutney; Dill Relish
4,Lurpak spreadable [Butters],Goat butter; Vaj - Enyhén sós; Dairypak; Maslac; Maslac,Vaj - Enyhén sós; Sviestas; Unt de masa; Maslac; Maslac
5,salami [Salami],Szalámi; Báthory csemege szalámi; Pick mangalica; Szarvas szalámi - csemege; salamisticks,Szalámi; Salami; Salami; Bauern Salami; Kulen


## 7. Latency (desktop CPU; not phone timing)
Single-text encoding of an uncatalogued product, and exact top-50 search versus corpus size (larger corpora are random unit vectors of the same dimension; search cost does not depend on content).

In [12]:
from sentence_transformers import SentenceTransformer
hf, rev, prefix = R.MODELS[chosen]
st = SentenceTransformer(hf, revision=rev, device='cpu')
sample_texts = [texts['full'][i] for i in rng.sample(range(N), 50)]
it = iter(sample_texts * 3)
lat = {'encode_single_torch': R.timed(lambda: st.encode([prefix + next(it)], normalize_embeddings=True), 100)}
dim = vf.shape[1]
for n in (1_000, 10_000, 100_000):
    M = vf[:n] if n <= N else R.normalize(np.random.default_rng(SEED).standard_normal((n, dim)))
    q = M[0].copy()
    lat[f'search_top50_n{n}'] = R.timed(lambda: np.argpartition(-(M @ q), 50)[:50], 200)
lat_df = pd.DataFrame(lat).T.round(2); lat_df

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

,median_ms,p95_ms
encode_single_torch,22.10,35.24
search_top50_n1000,0.05,0.07
search_top50_n10000,0.25,5.99
search_top50_n100000,8.98,28.56


## 8. ONNX export and parity
Export the chosen model to ONNX, then run it exactly as Java will (tokenizer.json → ONNX → mean pooling → L2) and compare against sentence-transformers. Gate: minimum cosine ≥ 0.999.

In [13]:
MODEL_DIR = R.export_onnx(chosen, ROOT / 'data/processed/models' / chosen)
onnx_enc = R.OnnxEncoder(MODEL_DIR, prefix)
parity_idx = rng.sample(range(N), 500)
ref = vf[parity_idx]; got = onnx_enc.encode([texts['full'][i] for i in parity_idx])
cos = (ref * got).sum(1)
it2 = iter(sample_texts * 3)
lat['encode_single_onnx'] = R.timed(lambda: onnx_enc.encode([next(it2)]), 100)
parity = {'n': len(cos), 'min_cosine': float(cos.min()), 'mean_cosine': float(cos.mean()), 'gate_pass': bool(cos.min() >= 0.999),
          'onnx_bytes': (MODEL_DIR / 'model.onnx').stat().st_size}
parity, lat['encode_single_onnx']

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/home/mih/cs/elfak/master/is/asap-app/.venv/lib/python3.13/site-packages/transformers/masking_utils.py:214: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if (padding_length := kv_length + kv_offset - attention_mask.shape[-1]) > 0:
/home/mih/cs/elfak/master/is/asap-app/.venv/lib/python3.13/site-packages/transformers/integrations/sdpa_attention.py:124: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  is_causal = q_length > 1 and attention_mask is None and is_causal


/home/mih/cs/elfak/master/is/asap-app/.venv/lib/python3.13/site-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset11.py:955: UserWarning: Exporting aten::index operator of advanced indexing in opset 17 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)


({'n': 500,
  'min_cosine': 0.9999998807907104,
  'mean_cosine': 1.0,
  'gate_pass': True,
  'onnx_bytes': 470233652},
 {'median_ms': 18.96692250011256, 'p95_ms': 32.99248010134761})

## 9. Export

In [14]:
np.save(ROOT / 'data/processed' / f'embeddings_{chosen}.npy', vf)
(ROOT / 'data/processed' / 'embedding_ids.json').write_text(json.dumps([p['id'] for p in cat]))
summary = {'env': env, 'dataset': stats, 'queries': len(queries), 'random_p@10': random_p10,
  'results': {f'{t}/{m}': {k: round(v, 4) for k, v in r.items()} for (t, m), r in res.iterrows()},
  'bootstrap_ndcg@10': {m: [round(x, 4) for x in v] for m, v in boot.items()},
  'encode_catalog_s': {f'{m}/{v}': round(s, 1) for (m, v), s in encode_s.items()},
  'chosen': chosen, 'chosen_revision': rev, 'beats_tfidf': beats_tfidf, 'latency_ms': {k: {a: round(b, 3) for a, b in v.items()} for k, v in lat.items()},
  'hybrid': hybrid_test, 'hybrid_beats_both': bool(hybrid_wins), 'hybrid_alpha_sweep': {str(k): v for k, v in sweep.round(4).to_dict().items()},
  'onnx_parity': parity, 'embeddings_sha256': hashlib.sha256(vf.tobytes()).hexdigest()}
(OUT / 'summary.json').write_text(json.dumps(summary, indent=2)); res.round(4).to_csv(OUT / 'retrieval_results.csv')
qual.to_csv(OUT / 'qualitative_top5.csv', index=False); pd.DataFrame(lat).T.round(3).to_csv(OUT / 'latency.csv')
print(json.dumps({k: summary[k] for k in ('chosen', 'beats_tfidf', 'onnx_parity')}, indent=1))

{
 "chosen": "e5-small",
 "beats_tfidf": false,
 "onnx_parity": {
  "n": 500,
  "min_cosine": 0.9999998807907104,
  "mean_cosine": 1.0,
  "gate_pass": true,
  "onnx_bytes": 470233652
 }
}


## Limitations
- Category agreement is a proxy for similarity; OFF categories are crowd-sourced and some leaf categories are near-duplicates, so absolute scores understate perceived quality. Relative ranking of methods is the decision signal.
- No human relevance judgements yet; a small judged set is a recommended addition before the final report.
- Timings are desktop CPU; phone-side and network latency are measured separately (S8).